In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

the model is not "looking" at what we want it to look at.  
I started out by blurring everything outside the area of interest, but the model started recognising texture (if the object inside the region of interest is smooth, the model chooses not to look at it). It cheated by becoming a texture detector.  

the next step is me multiplying the final layer outputs before the FC with a gaussian version of the clean mask.  

It is important to keep smooth objects separate in the training and validation set.  

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from sklearn.model_selection import train_test_split
from mtrain.neg_mask.model.datasets.blur_pad_dl import (
    make_box_of_crop_size_centered_at_box,
)
from mtrain.neg_mask.crops import get_region_crops
from mtrain.utils import DiskBooleanMask, DiskImage, show
from mtrain.denorm import denormalize_4chan_imagenet
from mtrain.neg_mask.model.datasets.blur_pad_dl import BlurPad4ChanDataset, CropTfmsOutsideBbox

# Run older model to see problems

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import (
    BlurPadGaussianDataset,
    BlurPadDataset,
)
from sklearn.model_selection import train_test_split

LABELS = ["other", "trash"]
IS_BLUR = True


B5P5_NOISY = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/b5p5_0pad_noisy"
)
DS_PATH = B5P5_NOISY

NUM_CHAN = 3


image_paths = list((DS_PATH / "train").glob("*.jpg"))
stratify = [BlurPadDataset.label_func(p) for p in image_paths]
train_paths, valid_paths = train_test_split(
    image_paths, test_size=0.2, stratify=stratify, random_state=42
)

In [ ]:
# should_noise = not IS_BLUR
# noise = 30 if should_noise else None

# force use noise now to make the smaller kernels jitter
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox, BlurPad4ChanDataset

noise = 30
print("using noise", noise)


MAX_NOISE = 30


def noise_overwriter(crop, crop_mask, inner_bbox):
    return CropTfmsOutsideBbox(crop, inner_bbox).overwrite_with_noise(MAX_NOISE).crop


train_ds = BlurPad4ChanDataset(train_paths, DS_PATH / "masks", 130, False, noise_overwriter)
valid_ds = BlurPad4ChanDataset(valid_paths, DS_PATH / "masks", 130, True, noise_overwriter)

In [ ]:
from fastai.vision.all import DataLoaders, default_device

dls = DataLoaders.from_dsets(
    train_ds,
    valid_ds,
    device=default_device(),
    num_workers=4,
    bs=16,
    persistent_workers=True,
)

In [ ]:
from fastai.callback.all import ProgressCallback
from torchvision.models import resnet18
from fastai.basics import F1Score, Precision, Recall, CrossEntropyLossFlat
from fastai.vision.all import vision_learner


def get_state_dict(path):
    # state_dict = torch.load("/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-p10-iter100-verified.pt", map_location=default_device())
    return torch.load(path, map_location=default_device())


def get_learner(dls):
    CLS_WEIGHT = torch.tensor([1.0, 2.5]).float().to("mps")
    learn = vision_learner(
        dls,
        resnet18,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        n_in=NUM_CHAN,
    )
    learn = learn.remove_cb(ProgressCallback)
    return learn


# MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-p10-iter100-verified.pt"
# MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/clean-with-subtract/fullset-p10-iter5.pt"
MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/0pad/fullset-iter30.pt"
learner = get_learner(dls)
state_dict = torch.load(MODEL_PATH, map_location=default_device())
learner.model.load_state_dict(state_dict, strict=True)

In [ ]:
preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
probs, targs, decoded, losses = preds
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]

In [ ]:
i = 2
idx = top_loss_idxs[i]

In [ ]:
from mtrain.denorm import denormalize_imagenet

img = denormalize_imagenet(learner.dls.valid_ds[i][0]).permute([1, 2, 0]).numpy()
plt.imshow(img)

# Data prep

## Create dataset of only clean photos

In [ ]:
import shutil
from mtrain.utils import mkdir, DiskImage, DiskBooleanMask
from tqdm import tqdm

CLEAN_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean"
)
TRAIN_DIR = CLEAN_DIR / "train"
IP_DS = CLEAN_DIR / "raw_dir_structure"


def create_train_ds(
    ip_ds,
    out_dir_root,
):
    # remove the out ds firs
    shutil.rmtree(out_dir_root, True)
    train_dir = mkdir(out_dir_root / "train")
    mask_dir = mkdir(out_dir_root / "masks")

    for label in ["other", "trash"]:
        label_d = ip_ds / label
        label_dirs = list(label_d.glob("*"))
        for d in tqdm(label_dirs):
            if not d.is_dir() or not (d / "orig.jpg").exists():
                continue

            fname = f"{label}_{d.name}"

            shutil.copy(d / "orig.jpg", train_dir / f"{fname}.jpg")
            shutil.copy(d / "mask.png", mask_dir / f"{fname}.png")


In [ ]:
create_train_ds(IP_DS, TRAIN_DIR)

## Test gaussian

In [ ]:
TRAIN_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/train"
)

In [ ]:
from mtrain.neg_mask.crops import padded_bbox
mask_path = TRAIN_DIR / "masks" / "other_03722627-0f39-46ac-b929-93ff5253b4b6_12.png"
image_path = TRAIN_DIR / "train" /  "other_03722627-0f39-46ac-b929-93ff5253b4b6_12.jpg"

mask = DiskBooleanMask.load(mask_path)
bbox = list(get_region_crops(mask))[0]
bbox = padded_bbox(bbox, 10, mask.shape)


bbox, inner_bbox = make_box_of_crop_size_centered_at_box(mask.shape, bbox, 128)

def random_tfm(cropped_image, inner_bbox):
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    # we can
    # overwrite with noise
    # step edge it
    # gaussian blur it
    # gaussian step down
    # noise overwrite is a fully killing operation

    # we can do blur + step down of any kind
    # what all do i want the model to see?
    # i want one fully noise cancelled photo
    # i want 2 different blurs atleast
    # i want one step down
    # i want one gaussian step down

    # i think the dataset can send all of them
    # or i can randomize
    NOISE_OVERWRITE = 0
    BLUR_1_7 = 1
    BLUR_2_13 = 2
    STEP_EDGE_0_3 = 3
    STEP_EDGE_0_7 = 4
    STEP_GAUSE_0_3 = 5

    i = random.randint(0, 5)
    add_noise_change = random.randint(0,2)
    def _add_noise(tfm):
        if add_noise_change == 1:
            return tfm.add_noise(30)
        return tfm


    if i == NOISE_OVERWRITE:
        return tfm.overwrite_with_noise(30).crop
    elif i == BLUR_1_7:
        tfm = tfm.overwrite_with_blur(7, 1)
        return _add_noise(tfm).crop
    elif i == BLUR_2_13:
        tfm = tfm.overwrite_with_blur(13, 2)
        return _add_noise(tfm).crop
    elif i == STEP_EDGE_0_3:
        tfm = tfm.step_down(0.3)
        return _add_noise(tfm).crop
    elif i == STEP_EDGE_0_7:
        tfm = tfm.step_down(0.7)
        return _add_noise(tfm).crop
    elif i == STEP_GAUSE_0_3:
        tfm = tfm.step_down_gaussian(0.3)
        return _add_noise(tfm).crop
    else:
        raise Exception(f"invalid i={i}")

    

    



image = DiskImage.load(image_path)
cropped_image = image[bbox.y : bbox.y2, bbox.x : bbox.x2]
orig = cropped_image.copy()

# tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
# orig = tfm.crop.copy()
# # blurred = tfm.step_down(0.5)
# blurred = tfm.overwrite_with_blur(13, 2)
blurred = random_tfm(cropped_image, inner_bbox)
# blurred = tfm.overwrite_with_noise(30)

show([orig, blurred])
# # cv2.rectangle(cropped_mask, (inner_bbox.x, inner_bbox.y), (inner_bbox.x2, inner_bbox.y2), 255, 3)
# print(inner_bbox)
# # plt.imshow(cropped_mask, cmap="gray")
# gs = mask_to_gaussian(cropped_mask, inner_bbox, 2.0, 0.3)

# plt.imshow(gs, cmap="gray")

In [ ]:
import torch
import torch.nn as nn


class MaskedResNet(nn.Module):
    def __init__(self, base_model, num_features=512):
        super().__init__()
        # Split ResNet into encoder (body) and head
        self.body = base_model[0]
        self.head = base_model[1]

        # LayerNorm to keep the model from "playing" with values
        # Assumes the last conv layer has 'num_features' channels
        self.bn = nn.BatchNorm2d(num_features, dtype=torch.float32)

    def forward(self, x):
        img = x[:,:3,:,:]  # first 3 chans image
        mask = x[:,3,:,:]  # last chan mask

        features = self.body(img)
        print(features.shape, mask.shape)
        normed = self.bn(features)
        # print(normed.shape, mask.shape)
        filtered = normed * mask
        return self.head(filtered)

In [ ]:

DS_PATH = TRAIN_DIR
image_paths = list((DS_PATH / "train").glob("*.jpg"))
stratify = [BlurPad4ChanDataset.label_func(p) for p in image_paths]
train_paths, valid_paths = train_test_split(
    image_paths, test_size=0.2, stratify=stratify, random_state=42
)

train_ds = BlurPad4ChanDataset(train_paths, DS_PATH / "masks", 130, False)
valid_ds = BlurPad4ChanDataset(valid_paths, DS_PATH / "masks", 130, True)

In [ ]:
train_ds[0][0].dtype

In [ ]:
image, mask = denormalize_4chan_imagenet(train_ds[25][0])
image = image.permute([1,2,0]).numpy()
mask = mask.numpy()

show([image, mask])

In [ ]:
from fastai.vision.all import unet_learner
unet_learner

In [ ]:
learner.model

In [ ]:
from fastai.vision.all import xresnet18, create_vision_model, Learner, DataLoaders, default_device, CrossEntropyLossFlat, ProgressCallback

dls = DataLoaders.from_dsets(
    train_ds,
    valid_ds,
    device=default_device(),
    num_workers=4,
    bs=16,
    persistent_workers=True,
)
weights = torch.tensor([1.0, 3.0], dtype=torch.float32).to(default_device())
model = create_vision_model(xresnet18, n_in=3, n_out=2, pretrained=True)
model = MaskedResNet(model)
learner = Learner(dls, model, loss_func=CrossEntropyLossFlat(weights))
learner = learner.remove_cb(ProgressCallback)

In [ ]:
from mtrain.example_dir.core import get_default_smallnet_50x50_learner
from fastai.vision.all import unet_learner, load_learner


smle = get_default_smallnet_50x50_learner()

smle.model

In [ ]:
learner.fine_tune(1)

In [ ]:
learner.fine_tune()

In [ ]:
tens = dls.one_batch()[0]

In [ ]:
tens[:,:2,:,:].shape, tens.shape